# Data Cleaning Challenge
## A reproducible, step-by-step data cleaning project

**Goal:** Take a deliberately messy customer-orders dataset, identify quality problems, clean it with documented decisions, and export a reliable dataset.

**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn  
**Level:** Beginner–Intermediate  
**How to use:** Run the notebook from top to bottom in Google Colab. The dataset is generated inside the notebook, so no external file is required.

### Project workflow
1. Create and inspect a deliberately messy dataset.
2. Profile missing values, duplicates, invalid values, and inconsistent formats.
3. Apply documented cleaning rules.
4. Validate the cleaned data.
5. Visualize before/after quality and summarize the results.
6. Download the cleaned CSV and a cleaning report.


## 1. Import libraries

We use pandas for tabular data cleaning, NumPy for missing values, and plotting libraries for visual inspection. If needed, Colab will install them automatically (they are usually preinstalled).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

## 2. Create the deliberately messy dataset

This simulated order dataset intentionally includes:
- Missing customer names, dates, quantities, and prices
- Duplicate order rows
- Inconsistent capitalization and extra spaces
- Mixed date formats and invalid dates
- Currency symbols and text mixed into numeric columns
- Invalid email addresses
- Negative quantities and impossible prices
- Inconsistent category labels

The data is synthetic and safe to experiment with.

In [ ]:
raw_data = [
    [1001, "  alice johnson ", "ALICE@example.com", "2024-01-05", "Electronics", "2", "$499.99", "Bengaluru"],
    [1002, "BOB SMITH", "bob.smith@email.com", "05/01/2024", " furniture ", "1", "₹12,500", "Mumbai"],
    [1003, "Charlie Brown", "charlie.email.com", "2024/01/07", "Grocery", "5", "25.50", "Delhi"],
    [1004, None, "dana@example.com", "2024-01-08", "Electronics", "1", "$899.00", "Chennai"],
    [1005, "Eva Green", "eva@example.com", "not-a-date", "Fashion", "3", "$45.00", "Hyderabad"],
    [1006, "Frank White", None, "2024-01-10", "fashion", "2", "$60.00", "Pune"],
    [1007, "Grace Lee", "grace.lee@example.com", "2024-01-11", "Grocery", "-2", "$10.00", "Kolkata"],
    [1008, "Henry King", "henry@example.com", "2024-01-12", "Electronics", "1", "-$99.00", "Bengaluru"],
    [1009, "Ivy Chen", "ivy@example.com", "2024-01-13", "Furniture", "two", "$300.00", "Mumbai"],
    [1010, "Jack Black", "jack@example.com", "2024-01-14", "Grocery", "4", "N/A", "Delhi"],
    [1011, "Kara Miles", " kara@example.com ", "14-01-2024", "FASHION", "1", "$75.00", "Chennai"],
    [1012, "Liam Scott", "liam@example.com", "2024-01-16", "Electronics", "2", "$1,299.00", "Hyderabad"],
    [1013, "Mia Davis", "mia@example.com", "2024-01-17", "Grocery", "3", "$8.50", "Pune"],
    [1014, "Noah Wilson", "noah@example.com", "2024-01-18", "Furniture", "1", "$450.00", "Kolkata"],
    [1014, "Noah Wilson", "noah@example.com", "2024-01-18", "Furniture", "1", "$450.00", "Kolkata"],
    [1015, "Olivia Taylor", "olivia@example.com", "2024-01-19", "Electronics", "1", "$250.00", "Bengaluru"],
    [1016, "Peter Adams", "peter@example.com", "2024-01-20", "Grocery", "0", "$12.00", "Mumbai"],
    [1017, "Quinn Baker", "quinn@example.com", "2024-01-21", "Home & Kitchen", "2", "$89.99", "Delhi"],
    [1018, "Riya Patel", "riya@example.com", "2024-01-22", "home and kitchen", "1", "$120.00", "Bengaluru"],
    [1019, "Sam Turner", "sam@example.com", "2024-01-23", "Fashion", "2", "$39.99", "Pune"],
    [1020, "Tina Moore", "tina@example.com", "2024-01-24", "Electronics", "1", "$699.00", "Chennai"],
    [1021, "Uma Shah", "uma@example.com", "2024-01-25", "Grocery", None, "$15.00", "Hyderabad"],
    [1022, "Victor Ray", "victor@example.com", "2024-01-26", "Furniture", "1", "$0", "Kolkata"],
    [1023, "Wendy Cruz", "wendy@example.com", "2024-01-27", "Electronics", "1", "$150.00", "Bengaluru"],
    [1024, "Xavier Hall", "xavier@example.com", "2024-01-28", "Fashion", "2", "$55.00", "Mumbai"],
    [1025, "Yara Khan", "yara@example.com", "2024-01-29", "Grocery", "3", "$7.50", "Delhi"],
    [1026, "Zane Ford", "zane@example.com", "2024-01-30", "Furniture", "1", "$220.00", "Pune"],
    [1027, "  AARAV MEHTA", "aarav@example.com", "2024-01-31", "electronics ", "1", "$349.00", "Bengaluru"],
    [1028, "Sara Ali", "sara@example.com", "2024-02-01", "Fashion", "1", "$89.00", "Chennai"],
    [1029, "Dev Kumar", "dev@example.com", "2024-02-02", "Grocery", "2", "$5.00", "Delhi"],
    [1030, "Nina Roy", "nina@example.com", "2024-02-03", "Furniture", "1", "$310.00", "Mumbai"],
]

columns = ["order_id", "customer_name", "email", "order_date", "category", "quantity", "unit_price", "city"]
df = pd.DataFrame(raw_data, columns=columns)

print(f"Raw dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head(10))

## 3. Initial data inspection

**Decision:** Never clean data blindly. First inspect shape, data types, sample rows, missing values, and duplicate rows. This establishes a baseline for the cleaning report.

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values per column:")
display(df.isna().sum().rename("missing_count").to_frame())

print("\\nExact duplicate rows:", df.duplicated().sum())
print("\\nSummary of original values:")
display(df.describe(include="all").T)

## 4. Create a baseline quality report

We record the original row count, exact duplicate count, and missing-value count so the impact of cleaning can be measured later.

In [ ]:
initial_rows = len(df)
initial_duplicates = int(df.duplicated().sum())
initial_missing_cells = int(df.isna().sum().sum())

quality_before = pd.DataFrame({
    "metric": ["Rows", "Columns", "Exact duplicate rows", "Missing cells"],
    "before_cleaning": [len(df), df.shape[1], initial_duplicates, initial_missing_cells]
})
display(quality_before)

## 5. Clean exact duplicate rows

**Decision 1:** Remove only rows that are identical across every column. We keep the first copy. We do not remove rows merely because one field repeats; two orders can legitimately share a customer or date.

In [ ]:
df_clean = df.copy()
duplicates_removed = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates(keep="first").copy()

print("Exact duplicate rows removed:", duplicates_removed)
print("Rows remaining:", len(df_clean))

## 6. Standardize text fields

**Decision 2:** Trim leading/trailing spaces and standardize customer names, emails, categories, and city names. Names use title case; email addresses use lowercase; categories and cities use consistent labels.

**Decision 3:** Map known category variants to one canonical category. Unknown categories are left unchanged after trimming/title-casing rather than guessed.

In [ ]:
# Remove leading/trailing whitespace and normalize case
for col in ["customer_name", "email", "category", "city"]:
    df_clean[col] = df_clean[col].astype("string").str.strip()

df_clean["customer_name"] = df_clean["customer_name"].str.title()
df_clean["email"] = df_clean["email"].str.lower()
df_clean["city"] = df_clean["city"].str.title()

category_map = {
    "electronics": "Electronics",
    "furniture": "Furniture",
    "grocery": "Grocery",
    "fashion": "Fashion",
    "home & kitchen": "Home & Kitchen",
    "home and kitchen": "Home & Kitchen"
}
df_clean["category"] = (
    df_clean["category"].str.lower().map(category_map)
    .fillna(df_clean["category"].str.title())
)

display(df_clean[["customer_name", "email", "category", "city"]].head(10))

## 7. Convert dates safely

**Decision 4:** Convert `order_date` to a real datetime value. The source contains several date formats, so we parse flexibly. Unparseable dates become missing (`NaT`) and are handled explicitly in the next step; we do not invent a date.

In [ ]:
df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"],
    errors="coerce",
    format="mixed",
    dayfirst=True
)

invalid_dates = int(df_clean["order_date"].isna().sum())
print("Missing or invalid dates after parsing:", invalid_dates)
display(df_clean[["order_id", "order_date"]].head(12))

## 8. Clean numeric columns

**Decision 5:** Convert quantity and unit price to numeric values. For price, remove currency symbols, commas, and other non-numeric characters while preserving decimal points and minus signs. Text such as `two` or `N/A` becomes missing rather than being guessed.

**Decision 6:** Quantities must be positive whole numbers. Unit prices must be greater than zero. Invalid values are marked missing.

In [ ]:
# Quantity: convert numeric-looking values; non-numeric strings become NaN
df_clean["quantity"] = pd.to_numeric(df_clean["quantity"], errors="coerce")

# Unit price: keep digits, decimal point, and minus sign
price_text = df_clean["unit_price"].astype("string")
price_text = price_text.str.replace(r"[^0-9.\-]", "", regex=True)
df_clean["unit_price"] = pd.to_numeric(price_text, errors="coerce")

# Enforce business rules
invalid_quantity_mask = (
    df_clean["quantity"].isna()
    | (df_clean["quantity"] <= 0)
    | (df_clean["quantity"] % 1 != 0)
)
invalid_price_mask = df_clean["unit_price"].isna() | (df_clean["unit_price"] <= 0)

invalid_quantities = int(invalid_quantity_mask.sum())
invalid_prices = int(invalid_price_mask.sum())

df_clean.loc[invalid_quantity_mask, "quantity"] = np.nan
df_clean.loc[invalid_price_mask, "unit_price"] = np.nan

display(df_clean[["order_id", "quantity", "unit_price"]].head(15))
print("Invalid quantities marked missing:", invalid_quantities)
print("Invalid prices marked missing:", invalid_prices)

## 9. Validate email addresses

**Decision 7:** A simple pattern check is used to identify obviously malformed email addresses. Invalid emails are set to missing. This is a basic format check, not proof that an email inbox exists.

In [ ]:
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
valid_email_mask = df_clean["email"].str.match(email_pattern, na=False)
invalid_emails = int((~valid_email_mask).sum())
df_clean.loc[~valid_email_mask, "email"] = pd.NA

print("Invalid or missing emails:", invalid_emails)
display(df_clean[["order_id", "email"]].head(12))

## 10. Handle missing values

**Decision 8:** Do not fill missing values with arbitrary constants.
- Rows missing a critical identifier (`order_id`) are not present in this dataset; if they were, they would need investigation.
- Rows missing `customer_name` or `order_date` are removed because they cannot be reliably attributed or placed in time.
- Missing quantity and unit price are retained as missing. We do not guess purchase amounts.
- Missing or invalid email is retained as missing because email is not required to calculate order value.

This policy preserves usable records without fabricating data.

In [ ]:
rows_before_required_drop = len(df_clean)
required_columns = ["customer_name", "order_date"]
df_clean = df_clean.dropna(subset=required_columns).copy()
rows_removed_missing_required = rows_before_required_drop - len(df_clean)

# Use nullable integer type for quantity
df_clean["quantity"] = df_clean["quantity"].astype("Int64")

# Derived field: calculated only when both inputs are present
df_clean["order_value"] = df_clean["quantity"] * df_clean["unit_price"]

print("Rows removed due to missing customer name/date:", rows_removed_missing_required)
print("Rows remaining:", len(df_clean))
display(df_clean.head(10))

## 11. Final validation

**Decision 9:** Re-check the cleaned data against the rules. The final dataset should have unique complete order IDs, valid dates, positive quantities/prices when present, standardized categories, and no exact duplicate rows.

In [ ]:
validation = {
    "duplicate_rows": int(df_clean.duplicated().sum()),
    "missing_order_ids": int(df_clean["order_id"].isna().sum()),
    "missing_customer_names": int(df_clean["customer_name"].isna().sum()),
    "missing_order_dates": int(df_clean["order_date"].isna().sum()),
    "non_positive_quantities": int((df_clean["quantity"].dropna() <= 0).sum()),
    "non_positive_prices": int((df_clean["unit_price"].dropna() <= 0).sum()),
    "invalid_email_formats": int((~df_clean["email"].str.match(email_pattern, na=False) & df_clean["email"].notna()).sum())
}
display(pd.Series(validation, name="count").to_frame())

assert validation["duplicate_rows"] == 0
assert validation["missing_order_ids"] == 0
assert validation["missing_customer_names"] == 0
assert validation["missing_order_dates"] == 0
assert validation["non_positive_quantities"] == 0
assert validation["non_positive_prices"] == 0
print("Core validation checks passed.")

## 12. Before-and-after quality comparison

This comparison summarizes how the cleaning process changed the dataset. Note that missing values in optional fields can remain intentionally.

In [ ]:
quality_after = {
    "Rows": len(df_clean),
    "Columns": df_clean.shape[1],
    "Exact duplicate rows": int(df_clean.duplicated().sum()),
    "Missing cells": int(df_clean.isna().sum().sum())
}

comparison = pd.DataFrame({
    "metric": ["Rows", "Columns", "Exact duplicate rows", "Missing cells"],
    "before_cleaning": [initial_rows, df.shape[1], initial_duplicates, initial_missing_cells],
    "after_cleaning": [quality_after["Rows"], quality_after["Columns"],
                       quality_after["Exact duplicate rows"], quality_after["Missing cells"]]
})
display(comparison)

## 13. Visualize data quality and cleaned data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=comparison.iloc[[0, 2, 3]], x="metric", y="before_cleaning", ax=axes[0], color="tomato")
axes[0].set_title("Before cleaning")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=comparison.iloc[[0, 2, 3]], x="metric", y="after_cleaning", ax=axes[1], color="seagreen")
axes[1].set_title("After cleaning")
axes[1].set_xlabel("")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
df_clean.groupby("category", dropna=False)["order_id"].count().sort_values(ascending=False).plot(kind="bar")
plt.title("Number of cleaned orders by category")
plt.xlabel("Category")
plt.ylabel("Number of orders")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
monthly = df_clean.dropna(subset=["order_date"]).set_index("order_date").resample("W")["order_id"].count()
monthly.plot(marker="o")
plt.title("Orders over time (weekly)")
plt.xlabel("Week")
plt.ylabel("Number of orders")
plt.tight_layout()
plt.show()

## 14. Review the final cleaned dataset

The `order_value` column is quantity × unit price. It remains missing if either input is missing, which is more honest than inventing a value.

In [ ]:
print(f"Final dataset: {len(df_clean)} rows × {df_clean.shape[1]} columns")
display(df_clean.sort_values("order_id").reset_index(drop=True))

print("\\nNumeric summary:")
display(df_clean[["quantity", "unit_price", "order_value"]].describe())

## 15. Export cleaned data and a repeatable cleaning report

Two files are created:
- `cleaned_customer_orders.csv` — the cleaned dataset.
- `data_cleaning_report.txt` — a plain-text record of the decisions and results.

In Google Colab, the next cell prompts downloads automatically.

In [ ]:
output_dir = Path("/content") if Path("/content").exists() else Path(".")
csv_path = output_dir / "cleaned_customer_orders.csv"
report_path = output_dir / "data_cleaning_report.txt"

df_clean.to_csv(csv_path, index=False)

report = f'''DATA CLEANING REPORT
====================

Project: Data Cleaning Challenge
Dataset: Synthetic customer orders

CLEANING DECISIONS
1. Removed exact duplicate rows; kept the first occurrence.
2. Trimmed whitespace from text fields.
3. Standardized customer names and city names to title case.
4. Standardized emails to lowercase.
5. Mapped known category variants to canonical labels.
6. Parsed dates; unparseable dates became missing.
7. Converted quantity and price to numeric values; invalid text became missing.
8. Required quantity to be a positive whole number.
9. Required unit price to be greater than zero.
10. Set malformed email addresses to missing (format check only).
11. Removed rows missing customer name or order date.
12. Kept missing optional numeric/email values missing; did not invent data.
13. Calculated order_value only when quantity and unit_price were available.

RESULTS
- Rows before cleaning: {initial_rows}
- Exact duplicates found and removed: {duplicates_removed}
- Rows removed for missing required fields: {rows_removed_missing_required}
- Rows after cleaning: {len(df_clean)}
- Missing values after cleaning: {int(df_clean.isna().sum().sum())}
- Invalid date values detected: {invalid_dates}
- Invalid quantities detected: {invalid_quantities}
- Invalid prices detected: {invalid_prices}
- Invalid email values detected: {invalid_emails}

VALIDATION
{json.dumps(validation, indent=2)}

LIMITATIONS
- Email validation checks format only; it does not confirm deliverability.
- This is a synthetic demonstration dataset.
- Business rules should be reviewed with the data owner before applying to real data.
'''

report_path.write_text(report, encoding="utf-8")

print("Saved files:")
print(csv_path)
print(report_path)
print("\\nCleaning report preview:\\n")
print(report)

In [ ]:
# Google Colab: download both output files
try:
    from google.colab import files
    files.download(str(csv_path))
    files.download(str(report_path))
except ImportError:
    print("Not running in Google Colab. Files are saved in the current working directory.")

## 16. Project conclusion

In this project, we transformed a deliberately messy order dataset into a more consistent and analysis-ready dataset.

**Key outcomes**
- Removed exact duplicate records.
- Standardized text and category values.
- Converted dates and numeric fields safely.
- Flagged invalid data instead of guessing replacements.
- Applied explicit rules for required and optional missing values.
- Validated the result and exported both the cleaned CSV and a decision report.

**Important lesson:** Data cleaning is not just code. Every transformation is a decision. Documenting the rule, reason, and impact makes the process reproducible and auditable.

### Suggested improvements
For a larger real-world project, add schema checks, logging, automated tests, source-data versioning, and a data dictionary approved by the data owner.